# 🏦 Home Credit Default Risk
## Phase 3B — Feature Engineering: Supplementary Tables (Layer 2)

> **Goal:** Aggregate all 6 supplementary tables into per-applicant features,
> merge them onto the main table, and measure AUC delta per table.
>
> **Prerequisite:** Run Phase 3A first and have `train_phase3a.parquet` available.

---

## 📦 0. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import gc
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline        import Pipeline
from sklearn.model_selection  import StratifiedKFold
from sklearn.metrics          import roc_auc_score
from sklearn.preprocessing    import OrdinalEncoder
from sklearn.base             import BaseEstimator, TransformerMixin

import lightgbm as lgb

RANDOM_STATE = 42
N_FOLDS      = 5
DATA_PATH    = './'

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
print('✅ Imports done')

## 1. Utility Functions

In [ ]:
def reduce_memory(df, verbose=True):
    """Downcast numeric columns to smallest valid dtype."""
    start_mem = df.memory_usage(deep=True).sum() / 1e6
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                for dtype in [np.int8, np.int16, np.int32]:
                    if c_min > np.iinfo(dtype).min and c_max < np.iinfo(dtype).max:
                        df[col] = df[col].astype(dtype)
                        break
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage(deep=True).sum() / 1e6
    if verbose:
        print(f'  Memory: {start_mem:.1f} MB → {end_mem:.1f} MB '
              f'({100*(start_mem-end_mem)/start_mem:.1f}% reduction)')
    return df


def safe_div(a, b, fill=0.0):
    """Division that handles zero denominator cleanly."""
    return np.where(b != 0, a / b, fill)


class CategoricalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoder   = OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1,
            encoded_missing_value=-2
        )
        self.cat_cols_ = []

    def fit(self, X, y=None):
        self.cat_cols_ = X.select_dtypes(include='object').columns.tolist()
        if self.cat_cols_:
            self.encoder.fit(X[self.cat_cols_])
        return self

    def transform(self, X):
        X = X.copy()
        if self.cat_cols_:
            X[self.cat_cols_] = self.encoder.transform(X[self.cat_cols_])
        return X


def run_cv(X, y, params, n_folds=5, label='experiment'):
    """Reusable stratified CV runner."""
    skf       = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    oof_preds = np.zeros(len(X))
    fold_aucs = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(100, verbose=False),
                lgb.log_evaluation(-1)
            ]
        )
        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        fold_aucs.append(roc_auc_score(y_val, oof_preds[val_idx]))

    oof_auc = roc_auc_score(y, oof_preds)
    print(f'  [{label}]  OOF AUC: {oof_auc:.6f}  |  '
          f'Mean: {np.mean(fold_aucs):.6f} ± {np.std(fold_aucs):.6f}')
    return oof_preds, oof_auc, fold_aucs


LGB_PARAMS = {
    'objective'        : 'binary',
    'metric'           : 'auc',
    'n_estimators'     : 2000,
    'learning_rate'    : 0.05,
    'num_leaves'       : 31,
    'min_child_samples': 20,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'reg_lambda'       : 1.0,
    'n_jobs'           : -1,
    'random_state'     : RANDOM_STATE,
    'verbose'          : -1,
}

print('✅ Utilities defined')

## 2. Load Phase 3A Output

In [ ]:
print('Loading Phase 3A engineered data...')
train = pd.read_parquet('train_phase3a.parquet')
test  = pd.read_parquet('test_phase3a.parquet')

y = train['TARGET'].copy()

print(f'  Train: {train.shape}')
print(f'  Test : {test.shape}')

# Experiment log — will grow as we add each table
experiment_log = []

# Encode categoricals on the base data first
encoder = CategoricalEncoder()
X_base  = train.drop(columns=['TARGET', 'SK_ID_CURR'], errors='ignore')
X_base  = encoder.fit_transform(X_base)

# Baseline AUC from Phase 3A
_, auc_3a, fold_aucs_3a = run_cv(X_base, y, LGB_PARAMS, label='phase3a_baseline')
experiment_log.append({
    'table': 'phase3a_baseline', 'new_features': 0,
    'total_features': X_base.shape[1],
    'oof_auc': round(auc_3a, 6), 'std': round(np.std(fold_aucs_3a), 6)
})
print(f'\n✅ Phase 3A baseline recorded: {auc_3a:.6f}')

---
## 3. Table 1 — `bureau.csv`

> Previous credits at **other financial institutions**.
> One row per external credit per applicant.

In [ ]:
def aggregate_bureau(path):
    """
    Aggregate bureau.csv to one row per SK_ID_CURR.

    Key questions being answered:
      - How many external credits does this person have?
      - What is their total current debt burden?
      - Have they ever overdue'd?
      - What's the health of their active credits?

    Subset features:
      All stats are computed twice — once on ALL credits,
      once on ACTIVE credits only. Active behavior is more
      predictive than historical behavior.
    """
    print('Loading bureau.csv...')
    bur = pd.read_csv(path + 'bureau.csv')
    bur = reduce_memory(bur)
    print(f'  Shape: {bur.shape}')

    # ── Encode CREDIT_ACTIVE ──────────────────────────────────────────────────
    bur['CREDIT_ACTIVE_BINARY'] = (bur['CREDIT_ACTIVE'] == 'Active').astype(np.int8)
    bur['CREDIT_OVERDUE_BINARY'] = (bur['CREDIT_DAY_OVERDUE'] > 0).astype(np.int8)

    # ── Derived fields ────────────────────────────────────────────────────────
    # How much of the credit has been used? (debt / limit)
    bur['BUREAU_CREDIT_UTILIZATION'] = safe_div(
        bur['AMT_CREDIT_SUM_DEBT'].fillna(0),
        bur['AMT_CREDIT_SUM'].fillna(1)
    )
    # How long until this credit ends?
    bur['BUREAU_CREDIT_DURATION'] = bur['DAYS_CREDIT_ENDDATE'] - bur['DAYS_CREDIT']

    # ── Aggregation — ALL credits ─────────────────────────────────────────────
    agg_all = bur.groupby('SK_ID_CURR').agg(
        BUREAU_LOAN_COUNT          = ('SK_ID_BUREAU',         'count'),
        BUREAU_ACTIVE_COUNT        = ('CREDIT_ACTIVE_BINARY', 'sum'),
        BUREAU_OVERDUE_COUNT       = ('CREDIT_OVERDUE_BINARY','sum'),
        BUREAU_PROLONG_COUNT       = ('CNT_CREDIT_PROLONG',   'sum'),

        # Credit amounts
        BUREAU_CREDIT_SUM_MEAN     = ('AMT_CREDIT_SUM',       'mean'),
        BUREAU_CREDIT_SUM_MAX      = ('AMT_CREDIT_SUM',       'max'),
        BUREAU_CREDIT_SUM_TOTAL    = ('AMT_CREDIT_SUM',       'sum'),

        # Debt
        BUREAU_DEBT_MEAN           = ('AMT_CREDIT_SUM_DEBT',  'mean'),
        BUREAU_DEBT_TOTAL          = ('AMT_CREDIT_SUM_DEBT',  'sum'),
        BUREAU_DEBT_MAX            = ('AMT_CREDIT_SUM_DEBT',  'max'),

        # Overdue amounts
        BUREAU_OVERDUE_AMT_MEAN    = ('AMT_CREDIT_SUM_OVERDUE','mean'),
        BUREAU_OVERDUE_AMT_MAX     = ('AMT_CREDIT_SUM_OVERDUE','max'),

        # Days past due
        BUREAU_DAY_OVERDUE_MAX     = ('CREDIT_DAY_OVERDUE',   'max'),
        BUREAU_DAY_OVERDUE_MEAN    = ('CREDIT_DAY_OVERDUE',   'mean'),

        # Utilization
        BUREAU_UTILIZATION_MEAN    = ('BUREAU_CREDIT_UTILIZATION','mean'),
        BUREAU_UTILIZATION_MAX     = ('BUREAU_CREDIT_UTILIZATION','max'),

        # Recency of credits
        BUREAU_DAYS_CREDIT_MEAN    = ('DAYS_CREDIT',          'mean'),
        BUREAU_DAYS_CREDIT_MAX     = ('DAYS_CREDIT',          'max'),   # Most recent
        BUREAU_DAYS_CREDIT_MIN     = ('DAYS_CREDIT',          'min'),   # Oldest
    ).reset_index()

    # ── Aggregation — ACTIVE credits only ─────────────────────────────────────
    bur_active = bur[bur['CREDIT_ACTIVE'] == 'Active']
    agg_active = bur_active.groupby('SK_ID_CURR').agg(
        BUREAU_ACTIVE_DEBT_TOTAL   = ('AMT_CREDIT_SUM_DEBT',  'sum'),
        BUREAU_ACTIVE_DEBT_MEAN    = ('AMT_CREDIT_SUM_DEBT',  'mean'),
        BUREAU_ACTIVE_CREDIT_MEAN  = ('AMT_CREDIT_SUM',       'mean'),
        BUREAU_ACTIVE_OVERDUE_MAX  = ('CREDIT_DAY_OVERDUE',   'max'),
        BUREAU_ACTIVE_UTIL_MEAN    = ('BUREAU_CREDIT_UTILIZATION','mean'),
    ).reset_index()
    agg_active.columns = ['SK_ID_CURR'] + [c for c in agg_active.columns[1:]]

    # ── Derived summary features ──────────────────────────────────────────────
    result = agg_all.merge(agg_active, on='SK_ID_CURR', how='left')

    # Active credit ratio
    result['BUREAU_ACTIVE_RATIO'] = safe_div(
        result['BUREAU_ACTIVE_COUNT'], result['BUREAU_LOAN_COUNT']
    )
    # Overdue credit ratio
    result['BUREAU_OVERDUE_RATIO'] = safe_div(
        result['BUREAU_OVERDUE_COUNT'], result['BUREAU_LOAN_COUNT']
    )
    # Debt-to-credit ratio across all bureau
    result['BUREAU_DEBT_CREDIT_RATIO'] = safe_div(
        result['BUREAU_DEBT_TOTAL'], result['BUREAU_CREDIT_SUM_TOTAL']
    )

    result = reduce_memory(result, verbose=False)
    del bur, bur_active, agg_all, agg_active
    gc.collect()

    print(f'  Bureau aggregation shape: {result.shape}')
    return result


bureau_agg = aggregate_bureau(DATA_PATH)
print(f'\nUnique applicants with bureau records: {len(bureau_agg):,}')
print(f'Applicants WITHOUT bureau records (no credit history): '
      f'{len(train) - bureau_agg["SK_ID_CURR"].isin(train["SK_ID_CURR"]).sum():,}')
print('→ NaN after merge = no credit history. This itself is a signal.')

---
## 4. Table 2 — `bureau_balance.csv`

> Monthly **status history** for each bureau credit.
> Two-level aggregation required.

In [ ]:
def aggregate_bureau_balance(path):
    """
    Aggregate bureau_balance.csv → SK_ID_CURR level.

    Two-level aggregation:
      1. SK_ID_BUREAU level (monthly records → one row per credit)
      2. SK_ID_CURR level  (credit records → one row per applicant)

    STATUS codes:
      C = closed, X = unknown,
      0 = no DPD, 1 = 1-30 DPD, 2 = 31-60, ..., 5 = 150+ DPD
      Higher number = worse. Treat numeric statuses as delinquency severity.
    """
    print('Loading bureau_balance.csv...')
    bb = pd.read_csv(path + 'bureau_balance.csv')
    bb = reduce_memory(bb)
    print(f'  Shape: {bb.shape}')

    # Convert STATUS to numeric delinquency (0=good, 5=worst, C/X=0)
    status_map = {'C': 0, 'X': 0, '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}
    bb['STATUS_NUM']  = bb['STATUS'].map(status_map).fillna(0).astype(np.int8)
    bb['IS_DPD']      = (bb['STATUS_NUM'] > 0).astype(np.int8)
    bb['IS_BAD']      = (bb['STATUS_NUM'] >= 2).astype(np.int8)  # 60+ DPD = serious

    # ── Level 1: aggregate by SK_ID_BUREAU ───────────────────────────────────
    bureau_stats = bb.groupby('SK_ID_BUREAU').agg(
        BB_MONTHS_COUNT     = ('MONTHS_BALANCE', 'count'),
        BB_STATUS_MEAN      = ('STATUS_NUM',      'mean'),
        BB_STATUS_MAX       = ('STATUS_NUM',      'max'),
        BB_DPD_COUNT        = ('IS_DPD',          'sum'),
        BB_BAD_COUNT        = ('IS_BAD',          'sum'),
    ).reset_index()

    bureau_stats['BB_DPD_RATE']  = safe_div(bureau_stats['BB_DPD_COUNT'],
                                             bureau_stats['BB_MONTHS_COUNT'])
    bureau_stats['BB_BAD_RATE']  = safe_div(bureau_stats['BB_BAD_COUNT'],
                                             bureau_stats['BB_MONTHS_COUNT'])

    # ── Level 2: join to bureau, aggregate by SK_ID_CURR ──────────────────────
    bureau_ids = pd.read_csv(path + 'bureau.csv', usecols=['SK_ID_CURR', 'SK_ID_BUREAU'])
    bureau_stats = bureau_stats.merge(bureau_ids, on='SK_ID_BUREAU', how='left')

    result = bureau_stats.groupby('SK_ID_CURR').agg(
        BB_NUM_BUREAU_CREDITS    = ('SK_ID_BUREAU',    'count'),
        BB_STATUS_MEAN_OF_MEANS  = ('BB_STATUS_MEAN',  'mean'),
        BB_STATUS_MAX_EVER       = ('BB_STATUS_MAX',   'max'),
        BB_TOTAL_DPD_MONTHS      = ('BB_DPD_COUNT',    'sum'),
        BB_TOTAL_BAD_MONTHS      = ('BB_BAD_COUNT',    'sum'),
        BB_DPD_RATE_MEAN         = ('BB_DPD_RATE',     'mean'),
        BB_BAD_RATE_MEAN         = ('BB_BAD_RATE',     'mean'),
        BB_DPD_RATE_MAX          = ('BB_DPD_RATE',     'max'),
    ).reset_index()

    # Prefix all new columns
    result = reduce_memory(result, verbose=False)
    del bb, bureau_stats, bureau_ids
    gc.collect()

    print(f'  Bureau balance aggregation shape: {result.shape}')
    return result


bureau_bal_agg = aggregate_bureau_balance(DATA_PATH)

---
## 5. Table 3 — `previous_application.csv`

> Past Home Credit loan **applications** — approved, refused, cancelled.

In [ ]:
def aggregate_previous_application(path):
    """
    Aggregate previous_application.csv to SK_ID_CURR level.

    Key questions:
      - How many times have they applied before?
      - What's their approval rate?
      - What were their previous loan characteristics?
      - Were they refused? What was the most recent refusal reason?

    Subset strategy:
      Compute stats for ALL, APPROVED only, and REFUSED only.
      Behavior on approved loans is more reliable than refused attempts.

    Real-world note:
      Application-to-credit ratio (how much they asked vs. how much they got)
      captures whether Home Credit has previously seen them as risky.
    """
    print('Loading previous_application.csv...')
    prev = pd.read_csv(path + 'previous_application.csv')
    prev = reduce_memory(prev)
    print(f'  Shape: {prev.shape}')

    # Fix sentinel values
    prev['DAYS_FIRST_DRAWING'].replace(365243, np.nan, inplace=True)
    prev['DAYS_FIRST_DUE'].replace(365243, np.nan, inplace=True)
    prev['DAYS_LAST_DUE_1ST_VERSION'].replace(365243, np.nan, inplace=True)
    prev['DAYS_LAST_DUE'].replace(365243, np.nan, inplace=True)
    prev['DAYS_TERMINATION'].replace(365243, np.nan, inplace=True)

    # Application-to-credit ratio: how much they asked vs. received
    prev['PREV_APP_CREDIT_RATIO']  = safe_div(prev['AMT_APPLICATION'],
                                               prev['AMT_CREDIT'].fillna(1))
    # Down payment as fraction of goods price
    prev['PREV_DOWN_PAYMENT_RATE'] = safe_div(prev['AMT_DOWN_PAYMENT'].fillna(0),
                                               prev['AMT_GOODS_PRICE'].fillna(1))
    # Is this a recent application?
    prev['PREV_IS_RECENT'] = (prev['DAYS_DECISION'] >= -365).astype(np.int8)

    # ── All previous applications ─────────────────────────────────────────────
    agg_all = prev.groupby('SK_ID_CURR').agg(
        PREV_APP_COUNT           = ('SK_ID_PREV',            'count'),
        PREV_AMT_CREDIT_MEAN     = ('AMT_CREDIT',            'mean'),
        PREV_AMT_CREDIT_MAX      = ('AMT_CREDIT',            'max'),
        PREV_AMT_ANNUITY_MEAN    = ('AMT_ANNUITY',           'mean'),
        PREV_AMT_APPLICATION_MEAN= ('AMT_APPLICATION',       'mean'),
        PREV_APP_CREDIT_RATIO_MEAN=('PREV_APP_CREDIT_RATIO', 'mean'),
        PREV_DOWN_PAYMENT_MEAN   = ('PREV_DOWN_PAYMENT_RATE','mean'),
        PREV_DAYS_DECISION_MAX   = ('DAYS_DECISION',         'max'),   # Most recent
        PREV_DAYS_DECISION_MEAN  = ('DAYS_DECISION',         'mean'),
        PREV_RECENT_COUNT        = ('PREV_IS_RECENT',        'sum'),
        PREV_CNT_PAYMENT_MEAN    = ('CNT_PAYMENT',           'mean'),  # Expected loan term
    ).reset_index()

    # ── Approved applications only ────────────────────────────────────────────
    approved = prev[prev['NAME_CONTRACT_STATUS'] == 'Approved']
    agg_approved = approved.groupby('SK_ID_CURR').agg(
        PREV_APPROVED_COUNT       = ('SK_ID_PREV',           'count'),
        PREV_APPROVED_CREDIT_MEAN = ('AMT_CREDIT',           'mean'),
        PREV_APPROVED_ANNUITY_MEAN= ('AMT_ANNUITY',          'mean'),
        PREV_APPROVED_RATIO_MEAN  = ('PREV_APP_CREDIT_RATIO','mean'),
    ).reset_index()

    # ── Refused applications only ─────────────────────────────────────────────
    refused = prev[prev['NAME_CONTRACT_STATUS'] == 'Refused']
    agg_refused = refused.groupby('SK_ID_CURR').agg(
        PREV_REFUSED_COUNT        = ('SK_ID_PREV',           'count'),
        PREV_REFUSED_CREDIT_MEAN  = ('AMT_CREDIT',           'mean'),
        PREV_REFUSED_DAYS_MEAN    = ('DAYS_DECISION',        'mean'),
    ).reset_index()

    # ── Merge all subsets ─────────────────────────────────────────────────────
    result = agg_all\
        .merge(agg_approved, on='SK_ID_CURR', how='left')\
        .merge(agg_refused,  on='SK_ID_CURR', how='left')

    # Approval and refusal rates
    result['PREV_APPROVAL_RATE'] = safe_div(
        result['PREV_APPROVED_COUNT'].fillna(0), result['PREV_APP_COUNT']
    )
    result['PREV_REFUSAL_RATE']  = safe_div(
        result['PREV_REFUSED_COUNT'].fillna(0),  result['PREV_APP_COUNT']
    )

    result = reduce_memory(result, verbose=False)
    del prev, approved, refused, agg_all, agg_approved, agg_refused
    gc.collect()

    print(f'  Previous application aggregation shape: {result.shape}')
    return result


prev_app_agg = aggregate_previous_application(DATA_PATH)

---
## 6. Table 4 — `installments_payments.csv`

> **Most behaviorally rich table.** Direct evidence of on-time vs. late payment.

In [ ]:
def aggregate_installments(path):
    """
    Aggregate installments_payments.csv to SK_ID_CURR level.

    This is the most behaviorally informative table:
      - Did they pay on time?
      - Did they pay the full amount?
      - Are they getting worse or better over time?

    Key derived features:
      DAYS_LATE = DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT
        Positive = late, Negative = early
      PAYMENT_RATIO = AMT_PAYMENT / AMT_INSTALMENT
        < 1 = underpaid, > 1 = overpaid, 1 = exact

    Real-world note:
      Payment behavior is the single strongest predictor of future default
      in consumer credit. A pattern of increasing lateness is a major flag.
    """
    print('Loading installments_payments.csv...')
    ins = pd.read_csv(path + 'installments_payments.csv')
    ins = reduce_memory(ins)
    print(f'  Shape: {ins.shape}')

    # ── Core derived features ─────────────────────────────────────────────────
    ins['DAYS_LATE']       = ins['DAYS_ENTRY_PAYMENT'] - ins['DAYS_INSTALMENT']
    ins['DAYS_EARLY']      = -ins['DAYS_LATE'].clip(upper=0)   # Positive when early
    ins['DAYS_LATE']       = ins['DAYS_LATE'].clip(lower=0)    # Positive when late

    ins['PAYMENT_RATIO']   = safe_div(ins['AMT_PAYMENT'].fillna(0),
                                       ins['AMT_INSTALMENT'].fillna(1))
    ins['PAYMENT_DIFF']    = ins['AMT_INSTALMENT'] - ins['AMT_PAYMENT'].fillna(0)

    ins['IS_LATE']         = (ins['DAYS_LATE'] > 0).astype(np.int8)
    ins['IS_VERY_LATE']    = (ins['DAYS_LATE'] > 30).astype(np.int8)
    ins['IS_UNDERPAID']    = (ins['PAYMENT_RATIO'] < 0.95).astype(np.int8)

    # ── Aggregation ───────────────────────────────────────────────────────────
    result = ins.groupby('SK_ID_CURR').agg(
        INS_PAYMENT_COUNT        = ('NUM_INSTALMENT_NUMBER',  'count'),

        # Lateness stats
        INS_DAYS_LATE_MEAN       = ('DAYS_LATE',              'mean'),
        INS_DAYS_LATE_MAX        = ('DAYS_LATE',              'max'),
        INS_DAYS_LATE_SUM        = ('DAYS_LATE',              'sum'),
        INS_DAYS_EARLY_MEAN      = ('DAYS_EARLY',             'mean'),

        # Late payment counts
        INS_LATE_COUNT           = ('IS_LATE',                'sum'),
        INS_VERY_LATE_COUNT      = ('IS_VERY_LATE',           'sum'),
        INS_UNDERPAID_COUNT      = ('IS_UNDERPAID',           'sum'),

        # Payment amount stats
        INS_PAYMENT_RATIO_MEAN   = ('PAYMENT_RATIO',          'mean'),
        INS_PAYMENT_RATIO_MIN    = ('PAYMENT_RATIO',          'min'),
        INS_PAYMENT_DIFF_MEAN    = ('PAYMENT_DIFF',           'mean'),
        INS_PAYMENT_DIFF_MAX     = ('PAYMENT_DIFF',           'max'),

        # Recency
        INS_DAYS_ENTRY_MAX       = ('DAYS_ENTRY_PAYMENT',     'max'),
    ).reset_index()

    # Derived rates
    result['INS_LATE_RATE']       = safe_div(result['INS_LATE_COUNT'],
                                              result['INS_PAYMENT_COUNT'])
    result['INS_VERY_LATE_RATE']  = safe_div(result['INS_VERY_LATE_COUNT'],
                                              result['INS_PAYMENT_COUNT'])
    result['INS_UNDERPAID_RATE']  = safe_div(result['INS_UNDERPAID_COUNT'],
                                              result['INS_PAYMENT_COUNT'])

    result = reduce_memory(result, verbose=False)
    del ins
    gc.collect()

    print(f'  Installments aggregation shape: {result.shape}')
    return result


ins_agg = aggregate_installments(DATA_PATH)

---
## 7. Table 5 — `POS_CASH_balance.csv`

> Monthly snapshots of **POS and cash loans** at Home Credit.

In [ ]:
def aggregate_pos_cash(path):
    """
    Aggregate POS_CASH_balance.csv to SK_ID_CURR level.

    SK_DPD = days past due for the month.
    SK_DPD_DEF = DPD for loans defined as defaulted.

    Key signal: Any months with DPD > 0 means they were late
    on an existing Home Credit loan — a direct behavioral signal.
    """
    print('Loading POS_CASH_balance.csv...')
    pos = pd.read_csv(path + 'POS_CASH_balance.csv')
    pos = reduce_memory(pos)
    print(f'  Shape: {pos.shape}')

    pos['IS_DPD']     = (pos['SK_DPD']     > 0).astype(np.int8)
    pos['IS_DPD_DEF'] = (pos['SK_DPD_DEF'] > 0).astype(np.int8)

    result = pos.groupby('SK_ID_CURR').agg(
        POS_MONTHS_COUNT       = ('MONTHS_BALANCE',         'count'),
        POS_SK_DPD_MEAN        = ('SK_DPD',                 'mean'),
        POS_SK_DPD_MAX         = ('SK_DPD',                 'max'),
        POS_SK_DPD_SUM         = ('SK_DPD',                 'sum'),
        POS_SK_DPD_DEF_MEAN    = ('SK_DPD_DEF',             'mean'),
        POS_SK_DPD_DEF_MAX     = ('SK_DPD_DEF',             'max'),
        POS_DPD_MONTH_COUNT    = ('IS_DPD',                 'sum'),
        POS_DPD_DEF_COUNT      = ('IS_DPD_DEF',             'sum'),
        POS_COMPLETED_COUNT    = ('NAME_CONTRACT_STATUS',
                                  lambda x: (x == 'Completed').sum()),
        POS_ACTIVE_COUNT       = ('NAME_CONTRACT_STATUS',
                                  lambda x: (x == 'Active').sum()),
        POS_CNT_INSTALMENT_MEAN= ('CNT_INSTALMENT',         'mean'),
        POS_CNT_INSTALMENT_FUTURE_MEAN = ('CNT_INSTALMENT_FUTURE', 'mean'),
    ).reset_index()

    result['POS_DPD_RATE']     = safe_div(result['POS_DPD_MONTH_COUNT'],
                                           result['POS_MONTHS_COUNT'])
    result['POS_COMPLETED_RATE']= safe_div(result['POS_COMPLETED_COUNT'],
                                            result['POS_MONTHS_COUNT'])

    result = reduce_memory(result, verbose=False)
    del pos
    gc.collect()

    print(f'  POS_CASH aggregation shape: {result.shape}')
    return result


pos_agg = aggregate_pos_cash(DATA_PATH)

---
## 8. Table 6 — `credit_card_balance.csv`

> Monthly **credit card balance** history at Home Credit.

In [ ]:
def aggregate_credit_card(path):
    """
    Aggregate credit_card_balance.csv to SK_ID_CURR level.

    Credit utilization rate is one of the most predictive signals
    in consumer credit scoring — it's a core component of FICO score.

    High utilization = financial stress = higher default risk.
    Maxed-out cards (utilization near 100%) are a serious red flag.

    Real-world note:
      Payment behavior patterns (only paying the minimum vs. paying in full)
      are heavily used in behavioral credit scoring models at banks.
    """
    print('Loading credit_card_balance.csv...')
    cc = pd.read_csv(path + 'credit_card_balance.csv')
    cc = reduce_memory(cc)
    print(f'  Shape: {cc.shape}')

    # ── Derived features ──────────────────────────────────────────────────────
    # Credit utilization: how much of their limit are they using?
    cc['CC_UTILIZATION']  = safe_div(
        cc['AMT_BALANCE'].fillna(0),
        cc['AMT_CREDIT_LIMIT_ACTUAL'].fillna(1)
    ).clip(0, 1)  # Cap at 100% (some values exceed limit)

    # Minimum payment ratio: are they only paying minimums?
    cc['CC_MIN_PAYMENT_RATIO'] = safe_div(
        cc['AMT_PAYMENT_CURRENT'].fillna(0),
        cc['AMT_INST_MIN_REGULARITY'].fillna(1)
    )

    # Drawing behavior: cash advances are riskier than regular purchases
    cc['CC_DRAWING_TOTAL'] = (
        cc['AMT_DRAWINGS_ATM_CURRENT'].fillna(0)   +
        cc['AMT_DRAWINGS_CURRENT'].fillna(0)        +
        cc['AMT_DRAWINGS_OTHER_CURRENT'].fillna(0)  +
        cc['AMT_DRAWINGS_POS_CURRENT'].fillna(0)
    )
    cc['CC_CASH_DRAW_RATIO'] = safe_div(
        cc['AMT_DRAWINGS_ATM_CURRENT'].fillna(0),
        cc['CC_DRAWING_TOTAL'].replace(0, np.nan)
    )

    cc['IS_DPD']    = (cc['SK_DPD']     > 0).astype(np.int8)
    cc['IS_MAXED']  = (cc['CC_UTILIZATION'] > 0.95).astype(np.int8)

    # ── Aggregation ───────────────────────────────────────────────────────────
    result = cc.groupby('SK_ID_CURR').agg(
        CC_MONTHS_COUNT            = ('MONTHS_BALANCE',         'count'),

        # Utilization
        CC_UTILIZATION_MEAN        = ('CC_UTILIZATION',         'mean'),
        CC_UTILIZATION_MAX         = ('CC_UTILIZATION',         'max'),
        CC_UTILIZATION_MIN         = ('CC_UTILIZATION',         'min'),

        # Balance
        CC_AMT_BALANCE_MEAN        = ('AMT_BALANCE',            'mean'),
        CC_AMT_BALANCE_MAX         = ('AMT_BALANCE',            'max'),

        # Payments
        CC_PAYMENT_MEAN            = ('AMT_PAYMENT_CURRENT',    'mean'),
        CC_MIN_PAYMENT_RATIO_MEAN  = ('CC_MIN_PAYMENT_RATIO',   'mean'),

        # Drawing behavior
        CC_DRAWING_TOTAL_MEAN      = ('CC_DRAWING_TOTAL',       'mean'),
        CC_CASH_DRAW_RATIO_MEAN    = ('CC_CASH_DRAW_RATIO',     'mean'),

        # DPD
        CC_DPD_MEAN                = ('SK_DPD',                 'mean'),
        CC_DPD_MAX                 = ('SK_DPD',                 'max'),
        CC_DPD_MONTH_COUNT         = ('IS_DPD',                 'sum'),
        CC_MAXED_COUNT             = ('IS_MAXED',               'sum'),

        # Credit limit
        CC_LIMIT_MEAN              = ('AMT_CREDIT_LIMIT_ACTUAL','mean'),
        CC_LIMIT_MAX               = ('AMT_CREDIT_LIMIT_ACTUAL','max'),
    ).reset_index()

    result['CC_DPD_RATE']   = safe_div(result['CC_DPD_MONTH_COUNT'],
                                        result['CC_MONTHS_COUNT'])
    result['CC_MAXED_RATE'] = safe_div(result['CC_MAXED_COUNT'],
                                        result['CC_MONTHS_COUNT'])

    result = reduce_memory(result, verbose=False)
    del cc
    gc.collect()

    print(f'  Credit card aggregation shape: {result.shape}')
    return result


cc_agg = aggregate_credit_card(DATA_PATH)

---
## 9. Merge All Tables & Measure AUC Per Table

In [ ]:
def merge_and_measure(base_train, base_test, agg_df, table_name, y, log, encoder):
    """
    Merge aggregated table onto train/test, run CV, record AUC delta.
    
    This is the core loop of Phase 3B:
    add one table → measure → decide whether it helps.
    """
    # Merge
    new_train = base_train.merge(agg_df, on='SK_ID_CURR', how='left')
    new_test  = base_test.merge(agg_df,  on='SK_ID_CURR', how='left')

    # Prepare X
    X = new_train.drop(columns=['TARGET', 'SK_ID_CURR'], errors='ignore')
    X = encoder.fit_transform(X)

    # Run CV
    _, auc, fold_aucs = run_cv(X, y, LGB_PARAMS, label=f'+{table_name}')

    prev_auc = log[-1]['oof_auc'] if log else 0
    log.append({
        'table'         : table_name,
        'new_features'  : agg_df.shape[1] - 1,  # minus SK_ID_CURR
        'total_features': X.shape[1],
        'oof_auc'       : round(auc, 6),
        'std'           : round(np.std(fold_aucs), 6),
        'auc_delta'     : round(auc - prev_auc, 6)
    })
    return new_train, new_test


# Reload base data (Phase 3A output)
train = pd.read_parquet('train_phase3a.parquet')
test  = pd.read_parquet('test_phase3a.parquet')
y     = train['TARGET'].copy()

encoder = CategoricalEncoder()

print('Merging tables and measuring AUC incrementally...\n')

# Add each table one at a time
train, test = merge_and_measure(train, test, bureau_agg,    'bureau',      y, experiment_log, encoder)
train, test = merge_and_measure(train, test, bureau_bal_agg,'bureau_bal',  y, experiment_log, encoder)
train, test = merge_and_measure(train, test, prev_app_agg,  'prev_app',    y, experiment_log, encoder)
train, test = merge_and_measure(train, test, ins_agg,       'installments',y, experiment_log, encoder)
train, test = merge_and_measure(train, test, pos_agg,       'pos_cash',    y, experiment_log, encoder)
train, test = merge_and_measure(train, test, cc_agg,        'credit_card', y, experiment_log, encoder)

print(f'\n✅ All tables merged')
print(f'Final train shape: {train.shape}')

## 10. Final Experiment Log & AUC Chart

In [ ]:
log_df = pd.DataFrame(experiment_log)

print('\n📋 Full Phase 3 Experiment Log:')
print(log_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# AUC progression
ax = axes[0]
ax.plot(log_df['table'], log_df['oof_auc'], 'o-', color='#3498db',
        linewidth=2.5, markersize=9)
ax.fill_between(range(len(log_df)),
                log_df['oof_auc'] - log_df['std'],
                log_df['oof_auc'] + log_df['std'],
                alpha=0.15, color='#3498db')
ax.set_xticks(range(len(log_df)))
ax.set_xticklabels(log_df['table'], rotation=30, ha='right')
ax.set_ylabel('OOF AUC')
ax.set_title('AUC Progression — All Tables', fontweight='bold')

# Delta per table
ax2 = axes[1]
deltas = log_df['auc_delta'].fillna(0)
colors_d = ['#2ecc71' if d >= 0 else '#e74c3c' for d in deltas]
ax2.bar(log_df['table'], deltas, color=colors_d)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xticks(range(len(log_df)))
ax2.set_xticklabels(log_df['table'], rotation=30, ha='right')
ax2.set_ylabel('AUC Delta')
ax2.set_title('AUC Gain per Table Added', fontweight='bold')

plt.tight_layout()
plt.show()

print('\n💡 Real-World Note:')
print('   Tables with negative AUC delta are still worth keeping if the delta')
print('   is within noise range (< std). Remove only if consistently negative')
print('   across multiple CV runs. Noise at this stage is normal.')

## 11. Feature Importance on Full Feature Set

In [ ]:
# Train final model on all features to get importance
X_final = train.drop(columns=['TARGET', 'SK_ID_CURR'], errors='ignore')
X_final = encoder.fit_transform(X_final)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
fi_all = pd.DataFrame()

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_final, y), 1):
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(
        X_final.iloc[tr_idx], y.iloc[tr_idx],
        eval_set=[(X_final.iloc[val_idx], y.iloc[val_idx])],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)]
    )
    fi_all = pd.concat([fi_all, pd.DataFrame({
        'feature': X_final.columns,
        'importance': model.feature_importances_,
        'fold': fold
    })])

mean_fi = fi_all.groupby('feature')['importance'].mean().sort_values(ascending=False)

# Plot top 40
top40 = mean_fi.head(40)
fig, ax = plt.subplots(figsize=(12, 12))

color_map = {
    'EXT_SOURCE': '#e74c3c',
    'BUREAU'    : '#3498db',
    'BB_'       : '#2980b9',
    'PREV_'     : '#2ecc71',
    'INS_'      : '#f39c12',
    'POS_'      : '#9b59b6',
    'CC_'       : '#1abc9c',
}
def get_color(feat):
    for prefix, color in color_map.items():
        if prefix in feat:
            return color
    return '#95a5a6'

bar_colors = [get_color(f) for f in top40.index[::-1]]
ax.barh(top40.index[::-1], top40.values[::-1], color=bar_colors)
ax.set_title('Top 40 Features by Importance (Phase 3 — All Tables)\n'
             '🔴 EXT_SOURCE  🔵 Bureau  🟢 Prev App  🟡 Installments  🟣 POS  🩵 Credit Card',
             fontweight='bold')
ax.set_xlabel('Mean Feature Importance')
plt.tight_layout()
plt.show()

print(f'\nTotal features: {X_final.shape[1]}')
print(f'Zero importance features: {(mean_fi == 0).sum()}')
print('\n→ Save top 40 feature list for Phase 4 feature selection experiments.')

## 12. Save Final Dataset

In [ ]:
# Save the complete engineered dataset for Phase 4 modeling
train.to_parquet('train_phase3_final.parquet', index=False)
test.to_parquet('test_phase3_final.parquet',   index=False)

# Save feature importance for Phase 4 reference
mean_fi.reset_index().to_csv('feature_importance_phase3.csv', index=False)

print(f'✅ Saved train_phase3_final.parquet : {train.shape}')
print(f'✅ Saved test_phase3_final.parquet  : {test.shape}')
print(f'✅ Saved feature_importance_phase3.csv')
print('\n→ Load these in Phase 4 for hyperparameter tuning and ensembling.')
print('\n📋 Summary:')
print(f'  Baseline AUC (Phase 2)   : ~0.755')
print(f'  Phase 3A AUC (app only)  : {experiment_log[0]["oof_auc"]}')
print(f'  Phase 3B AUC (all tables): {experiment_log[-1]["oof_auc"]}')
total_gain = experiment_log[-1]['oof_auc'] - 0.755
print(f'  Total AUC gain           : +{total_gain:.4f}')